# Set 06 – Hyperparameter-Tuning für Regression

Polynomgrad und Regularisierungsstärke sind Hyperparameter. Sie werden nicht beim Fitten gelernt, sondern durch Cross-Validation verglichen.

Der finale Testsatz bleibt bis zum Ende unberührt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(21)

## 1. Nichtlineare Regressionsdaten

Die Zielwerte folgen einer glatten nichtlinearen Funktion mit Rauschen.

In [ ]:
x = rng.uniform(-3.5, 3.5, 220)
y = 4 * np.sin(1.35 * x) + 0.45 * x**2 + rng.normal(0, 0.9, len(x))
X = pd.DataFrame({"x": x})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

ax = X_train.assign(y=y_train).plot.scatter(x="x", y="y", figsize=(8, 5), alpha=0.7)
ax.set_title("Trainingsdaten")
plt.show()

## 2. Pipeline definieren

Alle lernenden Schritte liegen in der Pipeline. Dadurch werden PolynomialFeatures und StandardScaler in jedem Cross-Validation-Fold ausschließlich aus dem jeweiligen Trainingsanteil gefittet.

In [ ]:
pipeline = Pipeline([
    ("polynom", PolynomialFeatures(include_bias=False)),
    ("skalierung", StandardScaler()),
    ("modell", Ridge()),
])

## 3. Suchraum festlegen

Wir vergleichen zwölf Polynomgrade und acht alpha-Werte. Das ergibt 96 Kombinationen. Jede Kombination wird in fünf Folds bewertet.

In [ ]:
parameter_grid = {
    "polynom__degree": list(range(1, 13)),
    "modell__alpha": np.logspace(-4, 3, 8),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
suche = GridSearchCV(
    estimator=pipeline,
    param_grid=parameter_grid,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)
suche.fit(X_train, y_train)

print("Beste Hyperparameter:", suche.best_params_)
print("Bester CV-RMSE:", round(np.sqrt(-suche.best_score_), 3))

## 4. Cross-Validation-Ergebnisse untersuchen

scikit-learn speichert negative Fehlerwerte, weil GridSearchCV immer größere Scores bevorzugt. Für die Darstellung wandeln wir sie zurück in RMSE.

In [ ]:
cv_ergebnisse = pd.DataFrame(suche.cv_results_)
cv_ergebnisse["CV_RMSE"] = np.sqrt(-cv_ergebnisse["mean_test_score"])
cv_ergebnisse["Train_RMSE"] = np.sqrt(-cv_ergebnisse["mean_train_score"])

spalten = [
    "param_polynom__degree",
    "param_modell__alpha",
    "Train_RMSE",
    "CV_RMSE",
    "rank_test_score",
]
display(cv_ergebnisse[spalten].sort_values("rank_test_score").head(12).round(4))

## 5. Fehlerlandschaft aus Polynomgrad und alpha

Dunkle Bereiche besitzen einen kleineren Cross-Validation-RMSE. Hohe Grade benötigen häufig stärkere Regularisierung, um stabil zu bleiben.

In [ ]:
matrix = cv_ergebnisse.pivot_table(
    index="param_polynom__degree",
    columns="param_modell__alpha",
    values="CV_RMSE",
)

fig, ax = plt.subplots(figsize=(11, 6))
bild = ax.imshow(matrix.values, aspect="auto", cmap="viridis_r")
ax.set_yticks(range(len(matrix.index)), matrix.index)
ax.set_xticks(range(len(matrix.columns)), [f"{wert:.0e}" for wert in matrix.columns], rotation=45)
ax.set_xlabel("Ridge alpha")
ax.set_ylabel("Polynomgrad")
ax.set_title("Cross-Validation-RMSE")
fig.colorbar(bild, ax=ax, label="RMSE")
plt.tight_layout()
plt.show()

## 6. Bestes Modell auf dem unangetasteten Testsatz

best_estimator_ wurde mit den besten Hyperparametern noch einmal auf allen Trainingsdaten gefittet. Erst jetzt verwenden wir die Testdaten.

In [ ]:
bestes_modell = suche.best_estimator_
y_test_pred = bestes_modell.predict(X_test)

mae = mean_absolute_error(y_test, y_test_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2 = r2_score(y_test, y_test_pred)

print(f"Test-MAE:  {mae:.3f}")
print(f"Test-RMSE: {rmse:.3f}")
print(f"Test-R²:   {r2:.3f}")

## 7. Gewählte Kurve und Residuen visualisieren

In [ ]:
x_gitter = np.linspace(X["x"].min(), X["x"].max(), 500)
X_gitter = pd.DataFrame({"x": x_gitter})
y_gitter = bestes_modell.predict(X_gitter)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_train["x"], y_train, alpha=0.45, label="Training")
axes[0].scatter(X_test["x"], y_test, alpha=0.85, label="Test")
axes[0].plot(x_gitter, y_gitter, color="black", linewidth=2.5, label="bestes Modell")
axes[0].set_title("Modell nach Cross-Validation")
axes[0].legend()

axes[1].scatter(y_test_pred, y_test - y_test_pred, color="#E45756")
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_xlabel("Vorhersage")
axes[1].set_ylabel("Residuum")
axes[1].set_title("Testresiduen")
plt.tight_layout()
plt.show()

## Sauberer Ablauf

1. Testsatz abtrennen.
2. Pipeline und Suchraum definieren.
3. Hyperparameter nur per Cross-Validation auf den Trainingsdaten auswählen.
4. best_estimator_ auf den Trainingsdaten verwenden.
5. Genau einmal final auf dem Testsatz bewerten.
6. Metriken, Residuen, Komplexität und Stabilität gemeinsam beurteilen.